In [2]:
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader
from torch.utils.tensorboard import  SummaryWriter
import einops
writer=SummaryWriter('logs')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
transform=torchvision.transforms.Compose([torchvision.transforms.RandAugment(num_ops=2,magnitude=6),torchvision.transforms.ToTensor()])
cifar_trainset=torchvision.datasets.CIFAR10('CIFAR10',train=True,transform=transform,download=True)
cifar_testset=torchvision.datasets.CIFAR10('CIFAR10',train=False,transform=torchvision.transforms.ToTensor(),download=True)
print(len(cifar_trainset))
cifar_trainset,cifar_validateset=torch.utils.data.random_split(cifar_trainset,[40000,10000])
load_trainset=DataLoader(cifar_trainset,batch_size=64,shuffle=True,drop_last=True)
load_validateset=DataLoader(cifar_validateset,batch_size=64,shuffle=True,drop_last=True)
load_testset=DataLoader(cifar_testset,batch_size=64,drop_last=False)

50000


In [5]:
class residual_connection(nn.Module):
    def __init__(self,in_channel,hidden_channel,out_channel,kernel_size=3,stride=1,padding=1):
        super().__init__()
        self.conv1=nn.Conv2d(in_channel,hidden_channel,kernel_size,stride,padding=padding)
        self.conv2=nn.Conv2d(hidden_channel,hidden_channel,kernel_size,padding=padding)
        self.conv3=nn.Conv2d(hidden_channel,out_channel,kernel_size,padding=padding)
        self.silu=nn.SiLU()
        self.batchnorm1=nn.BatchNorm2d(in_channel)
        self.batchnorm2=nn.BatchNorm2d(hidden_channel)
        self.batchnorm3=nn.BatchNorm2d(hidden_channel)
    def forward(self,x):
        y=x.clone()
        x=self.batchnorm1(x)
        x=self.conv1(x)
        x=self.silu(x)
        x=self.batchnorm2(x)
        x=self.conv2(x)
        x=self.silu(x)
        x=self.batchnorm3(x)
        x=self.conv3(x)
        x=self.silu(x)
        x=x+y
        return x
class resnetblock(nn.Module):
    def __init__(self,in_channel,hidden_channel,out_channel,num_residual_connection=2,kernel_size=3,stride=1,padding=1):
        super().__init__()
        self.residual_connections=nn.ModuleList([residual_connection(in_channel,hidden_channel,out_channel,kernel_size,stride,padding) for _ in range(num_residual_connection)])
    def forward(self,x):
        for residual_connection in self.residual_connections:
            x=residual_connection(x)
        return x
class resnet(nn.Module):
    def __init__(self):
        super().__init__()
        self.batchnorm1 = nn.BatchNorm2d(3)
        self.conv1 = nn.Conv2d(3, 32, 5, padding=2)
        self.resnetblock1=resnetblock(32,16,32,2,5,1,padding=2)
        self.pool1 = nn.MaxPool2d(2)
        self.silu1 = nn.SiLU()

        # self.batchnorm2 = nn.BatchNorm2d(32)
        # self.conv2 = nn.Conv2d(32, 32, 5, padding=2)
        self.resnetblock2=resnetblock(32,16,32,2,5,1,padding=2)
        self.pool2 = nn.MaxPool2d(2)
        self.silu2 = nn.SiLU()

        self.batchnorm3 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 64, 5, padding=2)
        self.resnetblock3=resnetblock(64,16,64,2,5,1,padding=2)
        self.pool3 = nn.MaxPool2d(2)
        self.silu3 = nn.SiLU()

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 16, 64)
        self.silu4 = nn.SiLU()
        self.fc2 = nn.Linear(64, 10)
    def forward(self,x):
        x=self.batchnorm1(x)
        x=self.conv1(x)
        x=self.resnetblock1(x)
        x=self.pool1(x)
        x=self.silu1(x)
        # x=self.batchnorm2(x)
        x=self.resnetblock2(x)
        x=self.pool2(x)
        x=self.silu2(x)
        x=self.batchnorm3(x)
        x=self.conv3(x)
        x=self.resnetblock3(x)
        x=self.pool3(x)
        x=self.silu3(x)
        x=self.flatten(x)
        x=self.silu4(self.fc1(x))
        x=self.fc2(x)
        return x
model=resnet()
model=model.to(device)
test=torch.randn(2,3,32,32,device=device)
# batchnormlayer=nn.BatchNorm2d(3)
# print(batchnormlayer(test))
print(model(test))

tensor([[-0.1387, -0.0112,  0.2309, -0.0117, -0.0049, -0.0280,  0.0466,  0.0499,
         -0.0571, -0.3858],
        [-0.3425,  0.0155,  0.2007, -0.0922, -0.0039, -0.0500,  0.0202, -0.0882,
         -0.0218, -0.2673]], device='cuda:0', grad_fn=<AddmmBackward0>)


In [104]:
class cifarmodel(nn.Module):
    def __init__(self):
        super().__init__()
        self.batchnorm1 = nn.BatchNorm2d(3)
        self.conv1 = nn.Conv2d(3, 32, 5, padding=2)
        self.pool1 = nn.MaxPool2d(2)
        self.silu1 = nn.SiLU()

        self.batchnorm2 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 32, 5, padding=2)
        self.pool2 = nn.MaxPool2d(2)
        self.silu2 = nn.SiLU()

        self.batchnorm3 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 64, 5, padding=2)
        self.pool3 = nn.MaxPool2d(2)
        self.silu3 = nn.SiLU()

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 16, 64)
        self.silu4 = nn.SiLU()
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.batchnorm1(x)
        x = self.pool1(self.conv1(x))
        x = self.silu1(x)

        x = self.batchnorm2(x)
        x = self.pool2(self.conv2(x))
        x = self.silu2(x)

        x = self.batchnorm3(x)
        x=self.conv3(x)
        x = self.pool3(x)
        x = self.silu3(x)

        x = self.flatten(x)
        x = self.silu4(self.fc1(x))
        x = self.fc2(x)
        return x

model=cifarmodel()
model=model.to(device)
test=torch.randn(2,3,32,32,device=device)
# batchnormlayer=nn.BatchNorm2d(3)
# print(batchnormlayer(test))
print(model(test))

tensor([[-0.1910,  0.0007,  0.0776, -0.0830,  0.0674,  0.0938,  0.0767, -0.2792,
         -0.0253, -0.0354],
        [-0.2289,  0.0396, -0.0196, -0.0069,  0.1502,  0.1267,  0.0609, -0.1837,
         -0.0170,  0.0028]], device='cuda:0', grad_fn=<AddmmBackward0>)


In [8]:
def get_metrics(model,load_testset,beta=1,fault_tolerance=0):
    acc=torch.zeros(10,device=device)
    predict=torch.zeros(10,device=device)
    total=torch.zeros(10,device=device)
    if fault_tolerance:
        for data in load_testset:
            imgs,label=data
            imgs=imgs.to(device)
            label=label.to(device)
            ans=model(imgs)
            for i in range(10):
                acc[i]+=torch.sum(torch.sum(torch.topk(ans,1+fault_tolerance,dim=1).indices==label.unsqueeze(1),dim=-1)*(label==i)).item()
                total[i]+=torch.sum(label==i).item()
                predict[i]+=torch.sum(torch.topk(ans,1+fault_tolerance,dim=1).indices==i).item()
    else:
        for data in load_testset:
            imgs,label=data
            imgs=imgs.to(device)
            label=label.to(device)
            ans=model(imgs)
            for i in range(10):
                acc[i]+=torch.sum((torch.argmax(ans,axis=1)==label)*(label==i)).item()
                total[i]+=torch.sum(label==i).item()
                predict[i]+=torch.sum(torch.argmax(ans,axis=1)==i).item()
    precision=acc/predict
    recall=acc/total
    f1=precision*recall/(beta**0.5*precision+recall)*(1+beta**0.5)
    accuracy=torch.sum(acc)/torch.sum(total)
    return f1,accuracy
print(get_metrics(model,load_testset,fault_tolerance=0))

(tensor([0.8098, 0.9090, 0.6923, 0.6276, 0.7471, 0.6964, 0.8385, 0.8058, 0.8947,
        0.8820], device='cuda:0'), tensor(0.7917, device='cuda:0'))


In [7]:
optimizer=torch.optim.AdamW(model.parameters(),lr=0.005)
loss=nn.CrossEntropyLoss()
loss=loss.to(device)
for epoch in range(20):
    totalloss=0
    for data in load_trainset:
        imgs,label=data
        imgs=imgs.to(device)
        label=label.to(device)
        result=model(imgs)
        result_loss=loss(result,label)
        optimizer.zero_grad()
        result_loss.backward()
        optimizer.step()
        totalloss+=result_loss.item()
    print(f"epoch={epoch},loss={totalloss},",end='')
    writer.add_scalar(tag='loss',scalar_value=totalloss,global_step=epoch)
    with torch.no_grad():
        f1,acc=get_metrics(model,load_validateset)
        print(f"f1={f1.mean()},acc={acc}")
        writer.add_scalar(tag='f1',scalar_value=f1.mean(),global_step=epoch)
        writer.add_scalar(tag='acc',scalar_value=acc,global_step=epoch)
writer.close()
torch.save(model,'cifar.pth')

epoch=0,loss=1106.7364794015884,f1=0.4614108204841614,acc=0.46754807233810425
epoch=1,loss=798.3447774648666,f1=0.593102216720581,acc=0.6017628312110901
epoch=2,loss=651.4418950676918,f1=0.6328725218772888,acc=0.6435296535491943
epoch=3,loss=575.5069551765919,f1=0.6776983737945557,acc=0.6778846383094788
epoch=4,loss=532.2559434175491,f1=0.6953796744346619,acc=0.6954126358032227
epoch=5,loss=492.7069199979305,f1=0.7104960680007935,acc=0.7136418223381042
epoch=6,loss=462.9493921995163,f1=0.7088608145713806,acc=0.7131410241127014
epoch=7,loss=437.70089614391327,f1=0.7385536432266235,acc=0.7389823794364929
epoch=8,loss=418.23529428243637,f1=0.7353461384773254,acc=0.7355769276618958
epoch=9,loss=398.7642153799534,f1=0.7416382431983948,acc=0.7448918223381042
epoch=10,loss=382.54418686032295,f1=0.7552027702331543,acc=0.754807710647583
epoch=11,loss=369.4561675488949,f1=0.7470885515213013,acc=0.7459936141967773
epoch=12,loss=358.4568723142147,f1=0.7423416972160339,acc=0.7442908883094788
epoch=

In [52]:
import einops
a=torch.randn(2,2,4,4)
print(a[0])
s=einops.rearrange(a,'batch channel (h patch1) (w patch2) -> batch channel h w (patch1 patch2)',patch1=2,patch2=2)
s=einops.rearrange(s,'batch channel h w patch -> batch (h w) (channel patch)')
print(s[0])


tensor([[[-2.0138e-01, -3.6401e-01, -6.6616e-01,  1.0251e+00],
         [ 2.9375e-01, -1.0303e+00,  1.8266e+00,  5.3554e-01],
         [ 3.0699e-01,  3.7983e-01,  1.2292e+00,  6.7285e-01],
         [ 4.8863e-01, -8.6254e-01, -1.7963e+00,  6.8612e-01]],

        [[ 3.6611e-01,  6.1031e-01, -1.7901e+00,  1.2595e+00],
         [ 2.2445e+00,  2.2237e+00, -1.1955e-03,  1.3158e+00],
         [ 5.6591e-01,  2.9753e-01, -3.2305e-01, -1.3151e-02],
         [ 5.1831e-01, -1.5531e+00, -1.0032e+00, -5.9396e-01]]])
tensor([[-2.0138e-01, -3.6401e-01,  2.9375e-01, -1.0303e+00,  3.6611e-01,
          6.1031e-01,  2.2445e+00,  2.2237e+00],
        [-6.6616e-01,  1.0251e+00,  1.8266e+00,  5.3554e-01, -1.7901e+00,
          1.2595e+00, -1.1955e-03,  1.3158e+00],
        [ 3.0699e-01,  3.7983e-01,  4.8863e-01, -8.6254e-01,  5.6591e-01,
          2.9753e-01,  5.1831e-01, -1.5531e+00],
        [ 1.2292e+00,  6.7285e-01, -1.7963e+00,  6.8612e-01, -3.2305e-01,
         -1.3151e-02, -1.0032e+00, -5.9396e-01]])

In [92]:
class ViT(nn.Module):
    def __init__(self,patch1=4,patch2=4,channel=3):
        super().__init__()
        self.patch1=patch1
        self.patch2=patch2
        self.tokenize=nn.Linear(patch1*patch2*channel,64)
        self.embedding=nn.parameter.Parameter(torch.randn(1,65,64,device=device),requires_grad=True)
        self.classembedding=nn.parameter.Parameter(torch.randn(64,device=device),requires_grad=True)
        self.transformer=nn.TransformerEncoder(nn.TransformerEncoderLayer(64,4,dim_feedforward=64,batch_first=True),num_layers=3)
        self.fc=nn.Linear(64,10)
    def forward(self,x):
        x=einops.rearrange(x,'batch channel (h patch1) (w patch2) -> batch channel h w (patch1 patch2)',patch1=self.patch1,patch2=self.patch2)
        x=einops.rearrange(x,'batch channel h w patch -> batch (h w) (channel patch)')
        x=self.tokenize(x)
        x=torch.cat((self.classembedding.unsqueeze(0).unsqueeze(0).expand(x.shape[0],-1,-1),x),dim=1)
        # print(x.shape)
        x=x+self.embedding
        x=self.transformer(x)
        x=self.fc(x[:,0])
        return x
model=ViT()
model=model.to(device)
test=torch.randn(2,3,32,32,device=device)
print(model(test).shape)

torch.Size([2, 10])


In [ ]:
optimizer=torch.optim.AdamW(model.parameters(),lr=0.001)
loss=nn.CrossEntropyLoss()
loss=loss.to(device)
for epoch in range(10):
    model.train()
    totalloss=0
    for data in load_trainset:
        imgs,label=data
        imgs=imgs.to(device)
        label=label.to(device)
        result=model(imgs)
        result_loss=loss(result,label)
        optimizer.zero_grad()
        result_loss.backward()
        optimizer.step()
        totalloss+=result_loss.item()
    print(f"epoch={epoch},loss={totalloss},",end='')
    writer.add_scalar(tag='loss',scalar_value=totalloss,global_step=epoch)
    model.eval()
    with torch.no_grad():
        f1,acc=get_metrics(model,load_validateset)
        print(f"f1={f1.mean()},acc={acc}")
        writer.add_scalar(tag='f1',scalar_value=f1.mean(),global_step=epoch)
        writer.add_scalar(tag='acc',scalar_value=acc,global_step=epoch)
writer.close()
torch.save(model,'cifar.pth')
print(get_metrics(model,load_testset,fault_tolerance=0))

epoch=0,loss=1391.5983583927155,f1=0.32348883152008057,acc=0.2042267620563507
epoch=1,loss=1317.6405928134918,f1=0.4319499135017395,acc=0.23788060247898102
epoch=2,loss=1296.5977786779404,f1=0.4765174984931946,acc=0.2578125
epoch=3,loss=1277.4139646291733,f1=0.49454960227012634,acc=0.26292067766189575
epoch=4,loss=1260.6025545597076,f1=0.5665228962898254,acc=0.2890625
epoch=5,loss=1246.9458063840866,f1=0.5688382387161255,acc=0.2959735691547394
epoch=6,loss=1234.7928332090378,f1=0.5797748565673828,acc=0.30548879504203796
epoch=7,loss=1216.713052034378,f1=0.6054840087890625,acc=0.3171073794364929
epoch=8,loss=1209.3576241731644,f1=0.6395444273948669,acc=0.3270232379436493
epoch=9,loss=1193.790199637413,f1=0.6333032846450806,acc=0.32652243971824646
(tensor([0.9141, 0.9260, 0.4015, 0.5255, 0.7489, 0.5703, 0.9057, 0.7602, 0.9498,
        0.9049], device='cuda:0'), tensor(0.3927, device='cuda:0'))
